# Suite DRV — The layout driver

`Driver` follows a diagram's journal, one entry at a time: a transaction's effect, which it sections into the logical layout's changes, anticipating what they remove, then applying it and calling the container layout and placement functions they imply, knowing nothing of the step that wrote it. Every bench suite runs through it; these cases are the entries no single step writes. mbse-basis [CAP-261: layout driven by any journal entry](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-261).

In [ ]:
import { assert, Bench, symbol, text } from "./support.js";
import { Container, Logical } from "../src/Framework/index.js";

const b = new Bench();
const L = b.source.Logical;

<a id="DRV-01"></a>
## DRV-01 · One entry that makes a top-level shape, a child of it, and sets the diagram's defaults: its grids configured first, then the defaults, then each shape placed, in the order the entry names them

In [ ]:
const d = symbol("");
assert(b.logical(["p", "p.q", "p.r"], (_: any) => _[d].grid_span(1n).grid_grows("columns")) === "[{\"kind\":\"grid\",\"container\":\"overview\",\"span\":1,\"grows\":\"columns\",\"cell_rotation\":0},{\"kind\":\"grid\",\"container\":\"p\",\"span\":1,\"grows\":\"columns\",\"cell_rotation\":0},{\"kind\":\"place\",\"shape\":\"p\",\"order\":0,\"to\":[0,0]},{\"kind\":\"extents\",\"rows\":1,\"columns\":1},{\"kind\":\"place\",\"shape\":\"q\",\"order\":0,\"to\":[0,0]},{\"kind\":\"extents\",\"rows\":1,\"columns\":1},{\"kind\":\"place\",\"shape\":\"r\",\"order\":1,\"to\":[0,1]},{\"kind\":\"extents\",\"rows\":1,\"columns\":2}]");
assert(text([b.cells(""), b.cells("p")]) === "[\"[[[0,0],[[\\\"p\\\",0]],null,1,1]]\",\"[[[0,0],[[\\\"q\\\",0]],null,1,1],[[0,1],[[\\\"r\\\",1]],null,1,1]]\"]");
assert(text(b.replay()) === "[]");

<a id="DRV-02"></a>
## DRV-02 · A spec turned or retargeted in place is unjoined before the entry and adjoined after it, by its new rotation or shapes

In [ ]:
assert(b.spec("p", "q", "r", 90.0) === "[{\"kind\":\"rule\",\"from_shape\":\"q\",\"to_shape\":\"r\",\"degrees\":90,\"status\":\"held\"}]");
const spec = { sp: L.OfAdjacency };
const turned = b.step("logical", ["turn the spec"], ["p.q"], spec,
  (_: any) => _[symbol("p.q")].owns({ "to r 90": (e: any) => e.owned(_.sp) }),
  (_: any) => _.sp.rotation((r: any) => r.degrees(270.0)));
assert(turned === "[{\"kind\":\"rule\",\"from_shape\":\"q\",\"to_shape\":\"r\",\"degrees\":90,\"status\":\"dropped\"},{\"kind\":\"move\",\"shape\":\"r\",\"from\":[0,1],\"to\":[0,0]},{\"kind\":\"move\",\"shape\":\"q\",\"from\":[0,0],\"to\":[0,1]},{\"kind\":\"rule\",\"from_shape\":\"q\",\"to_shape\":\"r\",\"degrees\":270,\"status\":\"met\"}]");
assert(text([b.cells("p"), b.rules("p")]) === "[\"[[[0,0],[[\\\"r\\\",1]],null,1,1],[[0,1],[[\\\"q\\\",0]],null,1,1]]\",\"[[\\\"q\\\",\\\"r\\\",270,false]]\"]");
assert(text(b.replay()) === "[]");
assert(b.child("p", "t") === "[{\"kind\":\"place\",\"shape\":\"t\",\"order\":2,\"to\":[0,2]},{\"kind\":\"extents\",\"rows\":1,\"columns\":3}]");
const retargeted = b.step("logical", ["retarget the spec"], ["p.q", "p.r", "p.t"], spec,
  (_: any) => _[symbol("p.q")].owns({ "to r 90": (e: any) => e.owned(_.sp) }),
  (_: any) => _.sp.to_shape((a: any) => a.remove((e: any) => e.shape(_[symbol("p.r")]))),
  (_: any) => _.sp.to_shape([_[symbol("p.t")]]));
assert(retargeted === "[{\"kind\":\"rule\",\"from_shape\":\"q\",\"to_shape\":\"r\",\"degrees\":270,\"status\":\"dropped\"},{\"kind\":\"move\",\"shape\":\"q\",\"from\":[0,1],\"to\":[0,0]},{\"kind\":\"move\",\"shape\":\"r\",\"from\":[0,0],\"to\":[0,1]},{\"kind\":\"move\",\"shape\":\"t\",\"from\":[0,2],\"to\":[0,0]},{\"kind\":\"move\",\"shape\":\"q\",\"from\":[0,0],\"to\":[0,1]},{\"kind\":\"move\",\"shape\":\"r\",\"from\":[0,1],\"to\":[0,2]},{\"kind\":\"rule\",\"from_shape\":\"q\",\"to_shape\":\"t\",\"degrees\":270,\"status\":\"met\"}]");
assert(text([b.cells("p"), b.rules("p")]) === "[\"[[[0,0],[[\\\"t\\\",2]],null,1,1],[[0,1],[[\\\"q\\\",0]],null,1,1],[[0,2],[[\\\"r\\\",1]],null,1,1]]\",\"[[\\\"q\\\",\\\"t\\\",270,false]]\"]");
assert(text(b.replay()) === "[]");

<a id="DRV-03"></a>
## DRV-03 · What it cannot follow it leaves: a member of a diagram with no grid is not placed, nor a spec between shapes with no cells, and nothing is refused; a layout store without the physical layout's schemas has no placement

In [ ]:
const lone = new Bench();
const s = symbol("");
assert(lone.step("logical", ["a member, no grid"], ["x", "y"], false,
  (_: any) => _[s].shapes([_[symbol("x")], _[symbol("y")]])) === "[]");
assert(lone.step("logical", ["a spec, no cells"], ["x", "y"], { sp: lone.source.Logical.OfAdjacency },
  (_: any) => _[symbol("x")].owns({ "to y 90": (e: any) => e.owned(_.sp) }),
  (_: any) => _.sp.from_shape([_[symbol("x")]]).to_shape([_[symbol("y")]])
    .rotation((r: any) => r.degrees(90.0))) === "[]");
assert(text([Logical.grid_of(lone.overview), Container.cell_of(lone.shape("x")), [...lone.shape("x").adjacent_to].length]) === "[null,null,1]");
assert(text(lone.replay()) === "[]");
assert(text([lone.driver.placement(lone.overview)]) === "[null]");